#### Working with DeltaTable, Delta files and DataFrames

- Save the DataFrame as delta files and read delta files into a DataFrame
- Update and delete the existing data using delta format

#### Command Reference

- Save a DataFrame in delta format files

  `df.write.format("delta").save(<path>)`


- Creating a DataFrame from Delta files

	`df = spark.read.format("delta").load(<path>)  `


- Create a DeltaTable from Delta files

	`from delta.tables import DeltaTable` <br/>
	`dt = DeltaTable.forPath(spark, <path>)`


- Create a DataFrame from a DeltaTable

	`df = dt.toDF()`


- Creating a SQL table on top of delta format files

	`CREATE TABLE db1.students(........)	USING DELTA	LOCATION '<path>'`


- Writing a DataFrame into a SQL delta table

	`df.write.insertInto("db1.students")`


- Create a DeltaTable from SQL delta table

	`from delta.tables import DeltaTable` <br/>
	`dt = DeltaTable.forName(spark, "db1.students")`

In [0]:
%run "../Includes/utils"

**Run the previous notebook to create the dataframes from well-formed JSON strings**

In [0]:
%run "./1. Create Sample Data"

In [0]:
display( students1_df )

In [0]:
display( students2_df )

**Save the dataframe into a directory as delta files**

In [0]:
dataPath = "/Volumes/workspace/default/data/delta/students"
dbutils.fs.rm(dataPath, True)

In [0]:
students1_df.write.format("delta").save(dataPath)

In [0]:
display(dbutils.fs.ls(dataPath))

**Read the delta files into a dataframe**

In [0]:
students_df = spark.read.format("delta").load(dataPath)
display(students_df)

**Create a DeltaTable reading from delta file path**

In [0]:
from delta.tables import DeltaTable
students_delta = DeltaTable.forPath(spark, dataPath)

In [0]:
#type(students_delta)

display(students_delta.toDF())

**Perform an UPDATE using the DeltaTable**

In [0]:
#help(students_delta)
help( students_delta.update)

In [0]:
type( students_delta )

In [0]:
students_delta.update(
    condition="student_id=4",
    set={"student_email": "'raju@gmail.com'"}
)

In [0]:
display(students_delta.toDF())

In [0]:
display(students_delta.history())

In [0]:
display(dbutils.fs.ls(dataPath))

In [0]:
students_delta.detail().display()

**Perform a DELETE using the DeltaTable**

In [0]:
students_delta.delete("student_id = 5")

In [0]:
#display(students_delta.toDF())
students_delta.toDF().display()

In [0]:
students_delta.history().display()

In [0]:
display( dbutils.fs.ls(dataPath) )

**DeltaTable & SQL**

In [0]:
dbutils.fs.rm(dataPath, True)

In [0]:
{"students": [{"student_id":1,"student_first_name":"Eduino","student_last_name":"Dawdry","student_email":"edawdry0@whitehouse.gov","student_gender":"Bigender","student_phone_numbers":["5737119029"],"student_address":{"street":"218 Ridgeway Crossing","city":"Omaha","state":"Nebraska","postal_code":"68110"}, "action": "I"}

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS students_db;
USE students_db;

In [0]:
%sql
DROP TABLE IF EXISTS students_db.students; 

CREATE OR REPLACE TABLE students_db.students (
  student_id LONG,
  student_first_name STRING,
  student_last_name STRING,
  student_email STRING,
  student_gender STRING,
  student_phone_numbers ARRAY<STRING>,
  student_address MAP<STRING,STRING>,
  action STRING
) 
USING DELTA

In [0]:
%sql
SELECT * FROM students_db.students

In [0]:
%sql
DESC EXTENDED  students_db.students

In [0]:
students1_str = """
{"students": [{"student_id":1,"student_first_name":"Eduino","student_last_name":"Dawdry","student_email":"edawdry0@whitehouse.gov","student_gender":"Bigender","student_phone_numbers":["5737119029"],"student_address":{"street":"218 Ridgeway Crossing","city":"Omaha","state":"Nebraska","postal_code":"68110"}, "action": "I"},
{"student_id":2,"student_first_name":"Lacee","student_last_name":"Prosek","student_email":"lprosek1@barnesandnoble.com","student_gender":"Polygender","student_phone_numbers":["9526294997","4699651256","7167123799","7061046839","7013761528"],"student_address":{"street":"188 Meadow Vale Avenue","city":"Augusta","state":"Georgia","postal_code":"30919"}, "action": "I"},
{"student_id":3,"student_first_name":"Richart","student_last_name":"Zimmer","student_email":"rzimmer2@ox.ac.uk","student_gender":"Non-binary","student_phone_numbers":["3129072019","2815879465","9793774370","6367833815"],"student_address":{"street":"87155 Lunder Court","city":"Fort Myers","state":"Florida","postal_code":"33994"}, "action": "I"},
{"student_id":4,"student_first_name":"Elyse","student_last_name":"Addionisio","student_email":"","student_gender":"Polygender","student_phone_numbers":["7347984926","3364474838","7136381150"],"student_address":{"street":"77 Sugar Alley","city":"Atlanta","state":"Georgia","postal_code":"31132"}, "action": "I"},
{"student_id":5,"student_first_name":"Lilian","student_last_name":"Warret","student_email":"","student_gender":"Male","student_phone_numbers":["5031246553","6151432197","2152754201"],"student_address":{"street":"82540 Summer Ridge Point","city":"Sioux Falls","state":"South Dakota","postal_code":"57193"}, "action": "I"}
]}
"""

In [0]:
import json
from pyspark.sql import Row

students1 = json.loads(students1_str)
students1_df = spark.createDataFrame([Row(**student) for student in students1['students']])
display(students1_df)

In [0]:
students1_df.write.insertInto("students_db.students")

In [0]:
%sql
SELECT * FROM students_db.students

In [0]:
from delta.tables import DeltaTable
students_delta = DeltaTable.forName(spark, "students_db.students")

In [0]:
display(students_delta.toDF())

In [0]:
students_delta.history().display()

In [0]:
%sql
DESC HISTORY students_db.students

In [0]:
students_delta.update(
  condition = "student_id = 4",
  set = {"student_email": "'raju@gmail.com'"}
)

In [0]:
students_delta.delete("student_id = 5")

In [0]:
students_delta.toDF().display()

In [0]:
%sql
SELECT * FROM students_db.students